# Эксперимент C2 — session-aware temporal-дообучение эмбеддера (Colab GPU)

Магистерская ВКР «Прикладной ИИ» (ТГУ). Ablation: поднимает ли **session-aware дообучение**
(K кадров особи из РАЗНЫХ сессий = cross-session позитивы) temporal re-ID Карелины (TK).

Это **эксперимент-доработка**: меряем на **dev (cross-fit OOF)**, sealed-тест НЕ трогаем,
официальные числа ВКР не меняются. Ноутбук **тонкий**: вся логика — в протестированном пакете `triton_crop`.

**Ablation-матрица** (один OOF-набор, один протокол; правка **A1**):
- **C0** — zero-shot (без обучения, референс);
- **C1** (`C1_naive_allsess`) — наивный сэмплер, но `train_all_sessions=True` (контроль с теми же данными);
- **C2** (`C2_session_allsess`) — + session-aware сэмплер, `train_all_sessions=True` (главный рычаг).

> **Зачем A1:** в первом прогоне у TK в обучающем пуле было 0 особей с ≥2 сессий (перепоимки лежат в
> probe) → session-aware был инертен. `train_all_sessions=True` берёт в обучение все сессии train-особей
> (анти-утечка по особям сохраняется) → у TK появляются cross-session пары, и C2 наконец может действовать.

## Локальные предпосылки (сделать ДО заливки архива)
1. `oof_records.csv` уже с колонкой `session` (перегенерирован ранее `embed-build`) — повторять НЕ нужно.
2. **Пересобрать архив** (в нём обновлённый `src/` с флагом `train_all_sessions`):
   `python -m triton_crop.cli embed-pack`  → залить `triton_block4_pack.zip` в любую папку `MyDrive`.
3. `Среда выполнения → Сменить среду → T4 GPU` → `Выполнить все`.

## Правила (уроки triton 2.0)
- `EPOCHS` не менять между resume одного прогона (cosine T = epochs).
- Сброс рантайма ~90 мин — норма: снова `Выполнить все`, resume подхватит готовые фолды из Drive.
- `freeze=3` (Swin-L) + grad-checkpointing уже включены → влезает в T4. При OOM — уменьшить `EPOCHS`/`batch_p`.
- **`test`/`open_test` НЕ вскрывать** — sealed-KPI заморожен. Здесь только dev-OOF на TK+PW.
- **LAB held-out** (генерализация на отдельную когорту) — опционально, после кропа LAB; см. финальную ячейку.

In [ ]:
# ════════ ПЕРЕКЛЮЧАТЕЛИ ════════
EPOCHS = 15        # 15 — быстро увидеть сигнал; 30 — финальный прогон. НЕ менять между resume одного прогона.
# Ablation-матрица (A1: C1/C2 учатся на ВСЕХ сессиях train-особей → у TK появляются cross-session пары).
# C0 zero-shot (epochs=0) · C1 наивный сэмплер · C2 session-aware. У C1/C2 train_all_sessions=True.
# Имена с _allsess → свежие чекпойнты (не путать с первым прогоном gallery-only).
CONFIGS = [
    {'name': 'C0_zeroshot',        'session_aware': False, 'epochs': 0,      'train_all_sessions': False},
    {'name': 'C1_naive_allsess',   'session_aware': False, 'epochs': EPOCHS, 'train_all_sessions': True},
    {'name': 'C2_session_allsess', 'session_aware': True,  'epochs': EPOCHS, 'train_all_sessions': True},
]

# 1. GPU + Google Drive (всё состояние — в Drive, переживает сброс рантайма)
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_DIR = '/content/drive/MyDrive/triton_temporal'
for c in CONFIGS:
    os.makedirs(f"{RUN_DIR}/ckpt/{c['name']}", exist_ok=True)
os.makedirs(RUN_DIR + '/oof', exist_ok=True)
print('configs:', [c['name'] for c in CONFIGS], '| EPOCHS:', EPOCHS, '| RUN_DIR:', RUN_DIR)

In [ ]:
# 2. Зависимости. НЕ переустанавливаем torch (колабовская CUDA-сборка — переустановка ломает CUDA).
!pip -q install timm==1.0.27 albumentations==2.0.8
import torch, timm
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| timm', timm.__version__)
assert torch.cuda.is_available(), 'Нет GPU: Среда выполнения -> Сменить среду -> T4 GPU'

In [ ]:
# 3. Распаковка пакета + подключение src + верификация целостности (анти-битый-аплоад по sha256)
from pathlib import Path
import glob, json, hashlib, sys
_p = (sorted(glob.glob('/content/drive/MyDrive/**/triton_temporal_pack.zip', recursive=True))
      or sorted(glob.glob('/content/drive/MyDrive/**/triton_block4_pack.zip', recursive=True)))
assert _p, 'архив *_pack.zip не найден в MyDrive — залей triton_block4_pack.zip (embed-pack)'
PACK = _p[0]; print('архив:', PACK)
!unzip -q -o "{PACK}" -d /content/work
ROOT = Path(sorted(glob.glob('/content/work/triton_*_pack*'))[0])
%cd {ROOT}
sys.path.insert(0, str(ROOT / 'src'))           # пакет берётся из src/ (Colab Python может быть < requires-python)
!pip -q install -e . 2>/dev/null || echo 'pip install пропущен — пакет подключён через sys.path (это нормально)'
man = json.loads((ROOT / 'artifacts/embed/pack_manifest.json').read_text(encoding='utf-8'))
bad = [p for p, h in man['png'].items()
       if not (ROOT / p).exists() or hashlib.sha256((ROOT / p).read_bytes()).hexdigest() != h]
assert not bad, f'битые/отсутствуют PNG: {bad[:5]} (всего {len(bad)})'
import triton_crop, triton_data
print('pack OK:', man['n_png'], 'PNG ·', man['n_records'], 'записей · пакеты подключены')

In [ ]:
# 4. Конфиг (MegaDescriptor-L-384, ribbon, freeze=3 из embed.yaml) + готовый OOF-набор (фолды по особям, seed=42)
import pandas as pd, numpy as np, torch
from PIL import Image
from dataclasses import replace
from triton_crop.config import load_embed_config

cfg = load_embed_config()                                   # дефолты финальной системы (Mega + ribbon)
cfg = replace(cfg, freeze_backbone_stages=3, batch_p=4, batch_k=4, epochs=EPOCHS)
print(cfg.base_model, '| freeze', cfg.freeze_backbone_stages, '| PxK', cfg.batch_p, 'x', cfg.batch_k,
      '| folds', cfg.cross_fit_folds)

oof = pd.read_csv('artifacts/embed/oof_records.csv')
oof = oof.dropna(subset=['path_belly_oriented', 'path_unroll_ribbon']).reset_index(drop=True)
assert 'session' in oof.columns, ('в oof_records.csv НЕТ колонки session — перегенерируй ЛОКАЛЬНО: '
    'python -m triton_crop.cli embed-build, затем embed-pack (см. предпосылки вверху ноутбука)')
print('OOF:', len(oof), 'кадров ·', dict(oof.role.value_counts()),
      '· сессий уникальных:', oof.session.nunique(),
      '· особь->1 fold:', bool((oof.groupby('individual_id').cross_fit_fold.nunique() == 1).all()))

def png_loader(path):
    return np.array(Image.open(ROOT / path).convert('RGB'))      # HWC uint8 — единый путь

def real_factory(name, device, num_classes=0):
    from triton_crop.embed_model import build_embedder
    e = build_embedder(name, device, num_classes=num_classes)
    m = e['model']
    if hasattr(m, 'set_grad_checkpointing'):                     # экономия VRAM на T4 (пересчёт активаций)
        m.set_grad_checkpointing(True)
    return e
torch.cuda.empty_cache()

In [ ]:
# 5. Прогон ablation: для каждого конфига — эмбеддинги OOF (cross-fit для C1/C2; zero-shot для C0) → npz в Drive.
#    Анти-утечка гарантируется кодом: каждый кадр эмбеддится моделью, не видевшей его особь.
from triton_crop.embed_train import cross_fit_embed, extract_embeddings
VARIANTS = ('belly_oriented', 'unroll_ribbon')
SESS = oof['session'].to_numpy()                                # результаты идут в порядке oof → session выровнен

def zeroshot_oof(cfg_i):
    """C0: zero-shot эмбеддинги (без обучения) — backbone базовой модели на всех кадрах oof."""
    e = real_factory(cfg_i.base_model, 'cuda')
    embedder = {'model': e['model'], 'transform': e['transform'], 'device': 'cuda'}
    out = {k: oof[k].to_numpy() for k in ('md5', 'role', 'individual_id', 'cohort')}
    for v in VARIANTS:
        out[v] = extract_embeddings(embedder, oof, f'path_{v}', png_loader)
    return out

results = {}
for c in CONFIGS:
    name = c['name']
    print(f"\n===== {name} (session_aware={c['session_aware']}, epochs={c['epochs']}, "
          f"train_all_sessions={c['train_all_sessions']}) =====", flush=True)
    cfg_i = replace(cfg, session_aware_sampling=c['session_aware'], epochs=c['epochs'],
                    train_all_sessions=c['train_all_sessions'])
    if c['epochs'] == 0:
        res = zeroshot_oof(cfg_i)
    else:
        res = cross_fit_embed(oof, cfg_i, build_embedder_fn=real_factory, image_loader=png_loader,
                              device='cuda', variants=VARIANTS, ckpt_dir=f"{RUN_DIR}/ckpt/{name}",
                              epochs=c['epochs'], resume=True)
    res['session'] = SESS                                       # выровнено по порядку oof
    results[name] = res
    np.savez(f"{RUN_DIR}/oof/{name}.npz",
             **{v: res[v] for v in VARIANTS}, role=res['role'], individual_id=res['individual_id'],
             session=res['session'], cohort=res['cohort'], md5=res['md5'])
    print(f"  {name}: сохранён {RUN_DIR}/oof/{name}.npz ·",
          {v: tuple(res[v].shape) for v in VARIANTS})
    torch.cuda.empty_cache()

In [ ]:
# 6. Предпросмотр сигнала прямо на Colab: TK temporal recall@1/5 по конфигам + парный McNemar против C0.
#    Авторитетный отчёт собирается ЛОКАЛЬНО, но это сразу показывает, есть ли сигнал у C2.
from triton_crop.embed_temporal import build_ablation_report
configs_oof = {c['name']: results[c['name']] for c in CONFIGS}
rep = build_ablation_report(configs_oof, variant='unroll_ribbon', cohort_filter='TK',
                            ks=tuple(cfg.recall_ks), primary_k=max(cfg.recall_ks))
print('=== TK temporal (dev OOF) ===')
for nm, blk in rep['per_config'].items():
    print(f"  {nm}: n={blk['n']} " + ' '.join(f"@{k}={blk.get('recall@%d' % k):.3f}" for k in cfg.recall_ks))
print('McNemar (vs C0, primary_k=%d):' % max(cfg.recall_ks))
for pair, mc in rep['mcnemar'].items():
    print(f"  {pair}: b={mc['b']} c={mc['c']} p={mc['p']:.4f}  (c>b ⇒ конфиг лучше базового)")
import json
json.dump(rep, open(f"{RUN_DIR}/oof/temporal_preview_TK.json", 'w', encoding='utf-8'),
          ensure_ascii=False, indent=2)
print('\nсохранён', RUN_DIR + '/oof/temporal_preview_TK.json')

## Дальше (локально) — авторитетный отчёт + гейт решения

1. Скачай `RUN_DIR/oof/{C0_zeroshot,C1_naive,C2_session}.npz` в `artifacts/embed/temporal_ablation/<name>/oof.npz`.
2. Собери отчёт (локальный шаг плана эксперимента) — `embed_temporal.build_ablation_report` на TK temporal (+ McNemar с поправкой Holm).
3. **🚦 Гейт решения:** C2 значимо бьёт C1/C0 на TK temporal @5? → да: следующий шаг (drift-ауг C3 + LAB в обучение C4); нет: честный null → future-work.

## Опционально — LAB held-out (генерализация на отдельную когорту)

Включается **после кропа LAB**. Нужно: (1) локально `crop --stages gallery,dev --scope temporal_aux --append`;
(2) собрать `lab_records.csv` (роль/сессия/пути вариантов) и добавить LAB-кропы в архив. Затем: для C0 — zero-shot эмбеддинг
LAB; для C1/C2 — усреднение эмбеддингов по фолд-моделям (LAB ни в одном фолде не участвовал → честный held-out);
сохранить `{name}_lab.npz` и прогнать `build_ablation_report(..., cohort_filter='LAB')`. Этот блок добавим, когда
LAB будет кропнут и упакован (по дисциплине анти-риска — после сигнала C2).